# WaveletSurfaceNet — train **and** run the full review-response suite on Colab, save everything to Drive

This notebook trains the mixed-base model **and** produces every piece of review-response evidence, writing
**all** of it to your Google Drive so nothing is lost on a disconnect:

1. **Train** the v2 mixed model (watertightness + noise/density augmentation).
2. **Before/after** v1 vs v2 (watertightness, noise, sparsity, SDF-error).
3. **Quantitative benchmark** over a ModelNet40 test sample — 6 methods (SPSR, BPA, APSS, RIMLS, tori, ours),
   F-score + **signed-distance error** + normal-consistency + mesh-defects + watertightness + time → `cmp_charts.png`.
4. **Qualitative** comparison grid → `cmp_baselines.png`.
5. **Gate-sensitivity** sweep (density / normal-noise / thresholds, ROC-AUC).
6. **Loss-term ablations** (optional; each is a short retrain).

Everything lands in `DRIVE_WS/` — checkpoints in `assets/`, figures in `paper/figs/`, tables/JSON/logs in
`results/`.  **Learned baselines** (POCO/NKSR/…) need their own legacy environments — see the last section.

**Before you start:** push your repo (the v2 `train.py` changes must be committed) — `git add -A && git commit && git push` —
or set `REPO_SOURCE="drive"`.  Runtime → **GPU** (T4 16 GB is enough; the comparison queries ours at R=128).

## 1· GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2· Configuration

In [ ]:
# --- code source ---
REPO_SOURCE = "github"          # "github" = clone ; "drive" = use a repo folder you uploaded to DRIVE_WS/repo
REPO_URL    = "https://github.com/OlegJakushkin/Points_as_supertoroids.git"
BRANCH      = "main"
DRIVE_WS    = "/content/drive/MyDrive/wsn_train"
LOCAL       = "/content/wsn"

# --- training ---
BASE, OUT   = "mixed", "waveshape_mixed_v2"
RESUME      = "assets/waveshape_mixed.pt"   # warm-start (v1); '' = from scratch
RES, BATCH, EPOCHS, REGION = 32, 24, 5, True

# --- which review-response steps to run after training ---
DO_BEFORE_AFTER = True      # check_v2.py (v1 vs v2)
DO_BENCHMARK    = True      # bench_val.py + fig_charts.py (quantitative, ModelNet40 sample)
DO_QUALITATIVE  = True      # fig_baselines.py (comparison grid)
DO_GATE_SWEEP   = True      # gate_sweep.py (gate sensitivity)
DO_ABLATIONS    = False     # loss-term ablations (each a short retrain -- slow; opt in)
VAL_K           = 3         # ModelNet40 shapes per category for the benchmark (3 -> 120 shapes)
GATE_PER        = 6         # shapes per category for the gate sweep
ABLATION_EPOCHS = 2
ABLATIONS = [               # (tag, extra train.py args) -- fine-tuned from the v1 checkpoint
    ("no_conn",   "--lam-conn 0"),
    ("no_sign",   "--lam-sign 0"),
    ("no_smooth", "--lam-smooth 0"),
    ("no_wave",   "--lam-wave 0"),
    ("no_seg",    "--no-seg"),
    ("signed_only",   "--base signed"),
    ("unsigned_only", "--base unsigned"),
]
print("config OK")

## 3· Mount Drive

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
RESULTS = f"{DRIVE_WS}/results"
os.makedirs(RESULTS, exist_ok=True)
print("workspace:", DRIVE_WS, "\nresults  :", RESULTS)

## 4· Get the code

In [ ]:
import os, shutil, subprocess
if REPO_SOURCE == "github":
    if os.path.isdir(LOCAL):
        subprocess.run(["git", "-C", LOCAL, "fetch", "--all"], check=True)
        subprocess.run(["git", "-C", LOCAL, "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", LOCAL, "pull", "origin", BRANCH], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, LOCAL], check=True)
elif REPO_SOURCE == "drive":
    src = f"{DRIVE_WS}/repo"
    assert os.path.isdir(src), f"upload your repo folder to {src} first"
    if os.path.isdir(LOCAL): shutil.rmtree(LOCAL)
    shutil.copytree(src, LOCAL, ignore=shutil.ignore_patterns('.git', 'data', 'renders', '__pycache__'))
os.chdir(LOCAL)
v2 = any("LAM_CONN = 0.6" in l for l in open("train.py"))
print("repo ready at", LOCAL, "| v2 train.py:", "YES" if v2 else "NO  <-- push your v2 changes!")

## 5· Redirect outputs to Drive
`assets/`, `data/`, `renders/`, `paper/figs/` become symlinks into Drive, so checkpoints, the cloud cache and
**every figure** are written straight to Drive (and reused next session). The committed `assets/*` (incl. the
v1 warm-start and the tori checkpoints) are seeded in once.

In [ ]:
import os, shutil
for sub in ["assets", "data", "renders", "paper/figs"]:
    dst = f"{DRIVE_WS}/{sub}"; os.makedirs(dst, exist_ok=True)
    src = f"{LOCAL}/{sub}"; os.makedirs(os.path.dirname(src), exist_ok=True)
    if os.path.islink(src):
        os.unlink(src)
    elif os.path.isdir(src):
        for f in os.listdir(src):
            s, d = f"{src}/{f}", f"{dst}/{f}"
            if not os.path.exists(d):
                (shutil.copytree if os.path.isdir(s) else shutil.copy2)(s, d)
        shutil.rmtree(src)
    os.symlink(dst, src)
    print(f"{sub:11s} -> {dst}")

## 6· Install dependencies (training **and** comparison baselines)

In [ ]:
# geometry + training libs
!pip install -q trimesh rtree scikit-image networkx scipy
# comparison baselines: SPSR/BPA via Open3D, APSS/RIMLS via PyMeshLab (needs the GL libs below), libigl
!apt-get -qq update > /dev/null && apt-get -qq install -y libopengl0 libegl1 libglu1-mesa libgl1 libgomp1 > /dev/null
!pip install -q open3d libigl pymeshlab
import trimesh, torch, open3d, pymeshlab
def _pmlver():
    try: return pymeshlab.version()                       # newer pymeshlab exposes a version() function
    except Exception: return getattr(pymeshlab, "__version__", "ok")
print("trimesh", trimesh.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| open3d", open3d.__version__, "| pymeshlab", _pmlver())

## 7· Build the training + evaluation data (cached in Drive, one time)

In [ ]:
import os, glob, zipfile, urllib.request
import numpy as np, torch
from waveshape import eval3d as E
from waveshape.datasets import load_mesh_normalized

DENSE, CAP = 1536, 9843
if not os.path.isdir("data/ModelNet40"):
    z = "data/ModelNet40.zip"
    if not os.path.exists(z):
        print("downloading ModelNet40 (~2GB, one time)...", flush=True)
        urllib.request.urlretrieve("http://modelnet.cs.princeton.edu/ModelNet40.zip", z)
    print("extracting...", flush=True)
    with zipfile.ZipFile(z) as zf: zf.extractall("data")

if not os.path.exists("data/se_clouds.pt"):
    files = sorted(glob.glob("data/ModelNet40/*/train/*.off"))[:CAP]
    print(f"building cloud cache from {len(files)} meshes...", flush=True)
    Ps, Ns = [], []
    for i, p in enumerate(files):
        try:
            m = load_mesh_normalized(p, max_faces=200000)
            Pp, Nn = E.sample_cloud(m, n=DENSE, noise=0.0, seed=0)
            if np.isfinite(Pp).all() and np.isfinite(Nn).all():
                Ps.append(Pp.astype(np.float32)); Ns.append(Nn.astype(np.float32))
        except Exception:
            continue
        if (i + 1) % 1000 == 0: print(f"  {i+1}/{len(files)} ({len(Ps)} ok)", flush=True)
    torch.save({"P": torch.tensor(np.stack(Ps)), "N": torch.tensor(np.stack(Ns))}, "data/se_clouds.pt")
    print(f"cached {len(Ps)} clouds -> data/se_clouds.pt", flush=True)
else:
    print("data/se_clouds.pt already cached in Drive", flush=True)

## 8· Train the v2 model
Auto-resumes from `assets/<OUT>_latest.pt` if present (survives disconnects). Checkpoints + renders are in Drive.

In [ ]:
import os
os.environ["MPLBACKEND"] = "Agg"
latest = f"assets/{OUT}_latest.pt"
resume = latest if os.path.exists(latest) else RESUME
args = f"--base {BASE} --out {OUT} --res {RES} --batch {BATCH} --epochs {EPOCHS}" + (" --region" if REGION else "")
if resume: args += f" --resume {resume}"
print("python train.py", args, flush=True)
!python train.py {args} 2>&1 | tee {RESULTS}/train_{OUT}.log

## 9· Before/after — v1 vs v2 (watertightness, noise, sparsity, SDF-error) → Drive
`check_v2.py` reconstructs the same shapes through both checkpoints and prints the deltas.

In [ ]:
if DO_BEFORE_AFTER and (os.path.exists(f"assets/{OUT}.pt") or os.path.exists(f"assets/{OUT}_latest.pt")):
    !MPLBACKEND=Agg PYTHONPATH=. python compare/check_v2.py 2>&1 | tee {RESULTS}/before_after_v1_vs_v2.txt
else:
    print("skipped (DO_BEFORE_AFTER off or no v2 checkpoint yet)")

## 10· Quantitative benchmark on ModelNet40 (6 methods, full metric set) → Drive
Runs SPSR / BPA / APSS / RIMLS / tori / **ours (v2)** over `VAL_K`×40 test shapes, computing F-score,
**signed-distance error**, normal-consistency, mesh-defects, watertightness and time, then draws `cmp_charts.png`.
This is the long step (≈30–60 min at `VAL_K=3`).

In [ ]:
import shutil
# v2's best-by-clean-SDF checkpoint may never beat v1 (v2 trades clean accuracy for robustness) -> use latest
CKPT = f"assets/{OUT}.pt" if os.path.exists(f"assets/{OUT}.pt") else f"assets/{OUT}_latest.pt"
if DO_BENCHMARK and os.path.exists(CKPT):
    !MIXED_CKPT={CKPT} VAL_K={VAL_K} MPLBACKEND=Agg PYTHONPATH=. python compare/bench_val.py 2>&1 | tee {RESULTS}/bench_val.log
    !MPLBACKEND=Agg PYTHONPATH=. python compare/fig_charts.py 2>&1 | tee {RESULTS}/fig_charts.log
    if os.path.exists("compare/metrics_val.json"): shutil.copy2("compare/metrics_val.json", RESULTS)
    print(f"benchmarked {CKPT} -> paper/figs/cmp_charts.png (Drive); metrics -> results/metrics_val.json")
else:
    print("skipped (DO_BENCHMARK off or no v2 checkpoint)")

## 11· Qualitative comparison grid → Drive

In [ ]:
import shutil
CKPT = f"assets/{OUT}.pt" if os.path.exists(f"assets/{OUT}.pt") else f"assets/{OUT}_latest.pt"
if DO_QUALITATIVE and os.path.exists(CKPT):
    !MIXED_CKPT={CKPT} MPLBACKEND=Agg PYTHONPATH=. python compare/fig_baselines.py 2>&1 | tee {RESULTS}/fig_baselines.log
    if os.path.exists("compare/metrics.json"): shutil.copy2("compare/metrics.json", RESULTS)
    from IPython.display import Image, display
    if os.path.exists("paper/figs/cmp_baselines.png"): display(Image("paper/figs/cmp_baselines.png"))
else:
    print("skipped")

## 12· Gate-sensitivity sweep → Drive
Gate accuracy vs point density, normal noise and the gate thresholds, with an ROC-AUC independent of the 0.30
operating point (the reviewer's *"no gate-accuracy evaluation / no threshold sensitivity"*). CPU-only.

In [ ]:
if DO_GATE_SWEEP:
    !GATE_PER={GATE_PER} PYTHONPATH=. python compare/gate_sweep.py 2>&1 | tee {RESULTS}/gate_sweep.txt
    import shutil
    if os.path.exists("compare/gate_sweep.json"): shutil.copy2("compare/gate_sweep.json", RESULTS)
else:
    print("skipped")

## 13· Loss-term ablations (optional — each is a short retrain) → Drive
Fine-tunes one variant per component (drop connectivity / sign / smooth / wavelet / seg; signed-only; unsigned-only)
from the v1 checkpoint for `ABLATION_EPOCHS`, so the held-out val SDF-error in each log shows that component's
marginal effect. Set `DO_ABLATIONS=True` and budget time (each variant ≈ `ABLATION_EPOCHS` epochs).

In [ ]:
if DO_ABLATIONS:
    for tag, extra in ABLATIONS:
        head = "" if "--base" in extra else "--base mixed "   # default mixed unless the ablation overrides the base
        a = f"{head}--out abl_{tag} --res {RES} --batch {BATCH} --epochs {ABLATION_EPOCHS} --region --resume {RESUME} {extra}"
        print(f"\n===== ablation {tag}: train.py {a} =====", flush=True)
        !MPLBACKEND=Agg python train.py {a} 2>&1 | tee {RESULTS}/ablation_{tag}.log
    print("\nablation checkpoints in assets/abl_*.pt ; val SDF-error is in each results/ablation_*.log")
else:
    print("skipped (set DO_ABLATIONS=True to run -- slow)")

## 14· Learned baselines — run `colab_baselines.ipynb`, then score them here

Neural-Pull / CAP-UDF / ConvONet need their own (conflicting) torch/CUDA envs, so they run in the companion
**`colab_baselines.ipynb`** (one method per session via condacolab) and write meshes to
`DRIVE_WS/results/learned/<method>/`. POCO and NKSR need heavy custom builds — run those via their Docker images
on a Docker host (`baselines_ext/<m>/`). The cell below scores whatever learned-baseline meshes are present in
Drive against our model and renders `cmp_learned.png` (per-shape, on the same 7 clouds shipped in the repo).
Neural-Pull / CAP-UDF are fair (per-shape); ConvONet is zero-shot/OOD — labelled `*`.

In [ ]:
import os
LEARNED_DIR = f"{DRIVE_WS}/results/learned"
CKPT = f"assets/{OUT}.pt" if os.path.exists(f"assets/{OUT}.pt") else f"assets/{OUT}_latest.pt"
have = [d for d in os.listdir(LEARNED_DIR) if os.path.isdir(f"{LEARNED_DIR}/{d}")] if os.path.isdir(LEARNED_DIR) else []
if have and os.path.exists(CKPT):
    print("learned baselines present:", have)
    !BL_OUT={LEARNED_DIR} MIXED_CKPT={CKPT} MPLBACKEND=Agg PYTHONPATH=. python compare/fig_learned.py 2>&1 | tee {RESULTS}/fig_learned.log
    from IPython.display import Image, display
    if os.path.exists("paper/figs/cmp_learned.png"): display(Image("paper/figs/cmp_learned.png"))
else:
    print(f"no learned-baseline meshes in {LEARNED_DIR} yet -- run colab_baselines.ipynb (one method per session) first")

In [ ]:
import glob, os, json, datetime
# tag this run so it composes with the host's results/local via compose.py
json.dump({"run": "colab", "model": OUT, "val_k": VAL_K,
           "date": datetime.datetime.now().isoformat(timespec="minutes")},
          open(f"{RESULTS}/meta.json", "w"), indent=1)
print("CHECKPOINTS:")
for p in sorted(glob.glob(f"{DRIVE_WS}/assets/{OUT}*.pt") + glob.glob(f"{DRIVE_WS}/assets/abl_*.pt")):
    print(f"  {p}  ({os.path.getsize(p)/1e6:.1f} MB)")
print("\nFIGURES (paper/figs):")
for p in sorted(glob.glob(f"{DRIVE_WS}/paper/figs/cmp_*.png")):
    print("  ", p)
print("\nRESULTS (compose these with the host's local run):")
for p in sorted(glob.glob(f"{RESULTS}/*")):
    print("  ", p)
print(f"\nTo compose: download the whole '{RESULTS}' folder, drop it into the repo as results/colab/,")
print("then run  python compose.py  ->  results/comparison.md  (ours@colab vs ours@local, fixed baselines).")

In [ ]:
import glob, os
print("CHECKPOINTS:")
for p in sorted(glob.glob(f"{DRIVE_WS}/assets/{OUT}*.pt") + glob.glob(f"{DRIVE_WS}/assets/abl_*.pt")):
    print(f"  {p}  ({os.path.getsize(p)/1e6:.1f} MB)")
print("\nFIGURES (paper/figs):")
for p in sorted(glob.glob(f"{DRIVE_WS}/paper/figs/cmp_*.png")):
    print("  ", p)
print("\nRESULTS (tables / JSON / logs):")
for p in sorted(glob.glob(f"{RESULTS}/*")):
    print("  ", p)